### FineTuning With Unsloth
- In this tutorial we'll finetune the Llama-3.2-3B-Instruct model using unsloth on the ServiceNow-AI/R1-Distill-SFT dataset to empower the llama model with DeepSeek-R1 like 'thinking' capabilities.
Let's start with installing the dependencies.

In [ ]:
!pip install -q unsloth
# Also get the latest nightly Unsloth!
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!pip install cuda-bindings==12.9.7

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# Use BF16 on Ampere+ GPUs (A100, L4, etc.), otherwise FP16 (T4, V100)
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Loaded: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Dtype: {DTYPE}")

- **model_name**:Specifies the name of the pre-trained model to load.
- **max_seq_length**:Defines the maximum sequence length (in tokens) that the model can process. max_seq_length = 2048 allows the model to process sequences up to 2048 tokens long.
- **dtype**:Specifies the data type for model weights and computations. None: Automatically selects the appropriate data type based on the hardware. torch.- - **float16**: Uses 16-bit floating point precision, reducing memory usage and potentially increasing speed on compatible GPUs. torch.bfloat16: Similar to float16 but with a wider dynamic range, beneficial for certain hardware like NVIDIA A100 GPUs.
- **load_in_4bit**:Determines whether to load the model using 4-bit quantization.Ideal for scenarios where memory efficiency is crucial, such as deploying models on edge devices or during experimentation.
Now, we'll use the get_peft_model from unsloth's FastLanguageModel class to attach adapters (peft layers) on top of the models in order to perform QLoRA

In [ ]:
from unsloth import FastLanguageModel

# LoRA configuration
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0
RANDOM_SEED = 3407

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA adapters attached successfully!")

- **r**: The rank of the low-rank matrices in LoRA; higher values can capture more information but increase memory usage.
- **target_modules**: List of model components (e.g., "q_proj", "k_proj") where LoRA adapters are inserted for fine-tuning.
- **lora_alpha**: Scaling factor for the LoRA updates; controls the impact of the adapters on the model's outputs.
- **lora_dropout**: Dropout rate applied to LoRA layers during training to prevent overfitting.
- **bias**: Specifies how biases are handled in LoRA layers; options include "none", "all", or "lora_only".
- **use_gradient_checkpointing**: Enables gradient checkpointing to reduce memory usage during training; "unsloth" uses Unsloth's optimized version.
- **random_state**: Seed for random number generators to ensure reproducibility of training results.
- **use_rslora**: Boolean indicating whether to use Rank-Stabilized LoRA (rsLoRA) for potentially more stable training.
-** ` loftq_config**: Configuration for Low-Rank Quantization (LoftQ); set to None to disable this feature.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ServiceNow-AI/R1-Distill-SFT",
    "v0",
    split="train"
)

In [ ]:
print(dataset[:5])

In [ ]:
EOS_TOKEN = tokenizer.eos_token

R1_PROMPT = """You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.

<problem>
{}
</problem>

{}
{}
"""

def formatting_prompts_func(examples):
    problems = examples["problem"]
    thoughts = examples["reannotated_assistant_content"]
    solutions = examples["solution"]

    texts = [
        R1_PROMPT.format(problem, thought, solution) + EOS_TOKEN
        for problem, thought, solution in zip(
            problems,
            thoughts,
            solutions
        )
    ]

    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Formatting dataset",
)

### Trainer Setup:

- **model and tokenizer**: These are the model and tokenizer objects that will be trained.

- **train_dataset**: The dataset used for training.

- **dataset_text_field**: Specifies the field in the dataset that contains the text data.

- **max_seq_length**: Maximum sequence length for the input data.

- **dataset_num_proc**: Number of processes to use for data loading.

- **packing**: If True, enables sequence packing (concatenates multiple examples into a single sequence to better utilize tokens).

### Training Arguments:

- **per_device_train_batch_size**: Number of samples per batch for each device.

- **gradient_accumulation_steps**: Number of steps to accumulate gradients before updating model weights.

- **warmup_steps**: Number of steps for learning rate warmup.

- **max_steps**: Total number of training steps.

- **learning_rate**: Learning rate for the optimizer.

- **fp16 and bf16**: Specifies whether to use 16-bit floating point precision or bfloat16, depending on hardware support.

- **logging_steps**: Frequency of logging training progress.

- **optim**: Optimizer type, here using an 8-bit version of AdamW.

- **weight_decay**: Regularization parameter for weight decay.

- **lr_scheduler_type**: Type of learning rate scheduler.

- **seed**: Random seed for reproducibility.

- **output_dir**: Directory where the training outputs will be saved.

- **report_to**: Integration for observability tools like "wandb", "tensorboard", etc.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

MAX_SEQ_LENGTH = 2048

training_args = TrainingArguments(
    output_dir="outputs",

    # Training
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,

    # Optimization
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="adamw_8bit",
    lr_scheduler_type="linear",

    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Logging
    logging_steps=1,
    report_to="none",

    # Checkpointing
    save_strategy="no",

    # Reproducibility
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Prompt
SYS_PROMPT = """You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.

<problem>
{}
</problem>
"""

question = "How many 'r's are present in 'strawberry'?"
message = SYS_PROMPT.format(question)

# Apply chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

# Enable optimized inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": message},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=1024,
    temperature=1.5,
    min_p=0.1,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print(response)

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

In [ ]:
# Save GGUF model
model.save_pretrained_gguf(
    "lora_model-001-3B-GGUF",
    tokenizer,
    quantization_method="q4_k_m",
)

### Running the model via Ollama (OPTIONAL)

In [ ]:
!apt-get update -qq

In [ ]:
!apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"])

import time
time.sleep(3)

In [ ]:
!curl http://127.0.0.1:11434/api/tags

In [ ]:
print(tokenizer._ollama_modelfile)

In [ ]:
!ollama create unsloth_model -f ./lora_model-001-3B-GGUF_gguf/Modelfile